# RACE Dataset — Exploratory Data Analysis (EDA)

## Overview
Analyzing the RACE (ReAding Comprehension from Examinations) dataset:
- **28,000+ passages** from Chinese school English exams
- **~100,000 questions** with 4 multiple-choice options (A/B/C/D)
- We've split into: 80% train (70,292), 10% val (8,787), 10% test (8,787)

This notebook explores:
1. Dataset structure and statistics
2. Passage lengths and distributions
3. Question characteristics
4. Answer balance and class distribution
5. Feature engineering needs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import os

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Load datasets
data_dir = "../data/processed"

train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
val_df = pd.read_csv(os.path.join(data_dir, "val.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))

print("✓ Datasets loaded!")
print(f"\nTrain: {train_df.shape[0]} rows")
print(f"Val:   {val_df.shape[0]} rows")
print(f"Test:  {test_df.shape[0]} rows")
print(f"Total: {train_df.shape[0] + val_df.shape[0] + test_df.shape[0]} rows")

In [ ]:
# Basic Info
print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)
print("\nTrain columns:", train_df.columns.tolist())
print("\nTrain sample (first row):")
print(train_df.iloc[0])
print("\nData types:")
print(train_df.dtypes)

In [ ]:
# Answer Distribution (Class Balance)
print("\n" + "=" * 60)
print("ANSWER DISTRIBUTION (CLASS BALANCE)")
print("=" * 60)

answer_counts_train = train_df['answer'].value_counts()
print("\nTrain answer distribution:")
print(answer_counts_train)
print(f"\nAnswer percentages:")
print((answer_counts_train / len(train_df) * 100).round(2))

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, (df, name) in enumerate([(train_df, "Train"), (val_df, "Val"), (test_df, "Test")]):
    answer_dist = df['answer'].value_counts()
    axes[idx].bar(answer_dist.index, answer_dist.values, color='steelblue')
    axes[idx].set_title(f"{name} - Answer Distribution")
    axes[idx].set_ylabel("Count")
    axes[idx].set_xlabel("Answer (A/B/C/D)")
    
plt.tight_layout()
plt.show()

print("✓ Answer distribution is fairly balanced across options")

In [ ]:
# Passage Length Analysis
print("\n" + "=" * 60)
print("PASSAGE LENGTH ANALYSIS")
print("=" * 60)

train_df['passage_length'] = train_df['article'].str.len()
train_df['passage_words'] = train_df['article'].str.split().str.len()

print("\nPassage length statistics (characters):")
print(train_df['passage_length'].describe())

print("\nPassage length statistics (words):")
print(train_df['passage_words'].describe())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df['passage_length'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title("Distribution of Passage Lengths (Characters)")
axes[0].set_xlabel("Character Count")
axes[0].set_ylabel("Frequency")

axes[1].hist(train_df['passage_words'], bins=50, color='steelblue', edgecolor='black')
axes[1].set_title("Distribution of Passage Lengths (Words)")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print(f"\nAverage passage: {train_df['passage_length'].mean():.0f} characters, {train_df['passage_words'].mean():.0f} words")

In [ ]:
# Question Analysis
print("\n" + "=" * 60)
print("QUESTION ANALYSIS")
print("=" * 60)

train_df['question_length'] = train_df['question'].str.len()
train_df['question_words'] = train_df['question'].str.split().str.len()

print("\nQuestion length statistics (characters):")
print(train_df['question_length'].describe())

print("\nQuestion length statistics (words):")
print(train_df['question_words'].describe())

# Check question starters (Wh-word types)
wh_words = ['who', 'what', 'where', 'when', 'why', 'how']
print("\nQuestion types (first word detection):")
for wh in wh_words:
    count = (train_df['question'].str.lower().str.startswith(wh)).sum()
    pct = count / len(train_df) * 100
    print(f"  {wh.capitalize()}: {count:5d} ({pct:5.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df['question_length'], bins=50, color='coral', edgecolor='black')
axes[0].set_title("Distribution of Question Lengths (Characters)")
axes[0].set_xlabel("Character Count")
axes[0].set_ylabel("Frequency")

axes[1].hist(train_df['question_words'], bins=30, color='coral', edgecolor='black')
axes[1].set_title("Distribution of Question Lengths (Words)")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Answer Option Analysis
print("\n" + "=" * 60)
print("ANSWER OPTIONS ANALYSIS")
print("=" * 60)

for opt in ['A', 'B', 'C', 'D']:
    train_df[f'option_{opt}_len'] = train_df[opt].str.len()
    train_df[f'option_{opt}_words'] = train_df[opt].str.split().str.len()

print("\nOption lengths (words) - Statistics:")
for opt in ['A', 'B', 'C', 'D']:
    lengths = train_df[f'option_{opt}_words']
    print(f"  Option {opt}: mean={lengths.mean():.1f}, std={lengths.std():.1f}, min={lengths.min()}, max={lengths.max()}")

# Check if correct answer is different across distributions
print("\nCorrect answer option distribution:")
for opt in ['A', 'B', 'C', 'D']:
    count = (train_df['answer'] == opt).sum()
    pct = count / len(train_df) * 100
    print(f"  Answer {opt}: {count:5d} ({pct:5.1f}%)")

# Visualize option lengths
fig, ax = plt.subplots(figsize=(12, 5))
option_data = [train_df[f'option_{opt}_words'].values for opt in ['A', 'B', 'C', 'D']]
ax.boxplot(option_data, labels=['A', 'B', 'C', 'D'])
ax.set_title("Distribution of Answer Option Lengths (Words) by Option")
ax.set_ylabel("Word Count")
ax.set_xlabel("Option")
plt.show()

## Key Insights & Next Steps

### Dataset Summary
- **Total: 87,866 samples** split into 80/10/10 (train/val/test)
- **Balanced classes**: Answers (A/B/C/D) are roughly equally distributed (~25% each)
- **Diverse passages**: Lengths vary significantly (short to very long articles)
- **Question types**: Mix of Wh-words (Who/What/Where/When/Why/How)

### Feature Engineering Considerations
1. **One-Hot Encoding** (primary):
   - Convert words to binary vectors (word present = 1, absent = 0)
   - Create matrices for: passages, questions, answer options
   
2. **Lexical Features**:
   - Passage length, question length, option length
   - Word overlap between question and passage
   - Vocabulary richness
   
3. **Text Preprocessing** (before encoding):
   - Lowercasing
   - Remove punctuation
   - Tokenization
   - Optional: stemming/lemmatization

### Model A (Answer Verification)
- Task: Predict if a given (passage, question, option) triple is correct
- Features: One-Hot vectors of combined passage + question + option
- Models: Logistic Regression, SVM, Random Forest

### Model B (Distractor & Hint Generation)
- Task 1: Generate wrong-but-plausible answer options
- Task 2: Extract graduated hints from passage
- Features: Cosine similarity between options using One-Hot encoding

### Evaluation Metrics (Per Professor)
- **BLEU Score**: Compare generated text to reference
- **ROUGE Score**: Overlap metrics between generated and reference
- **METEOR Score**: Semantic similarity of generated text

Next: Build feature engineering pipeline → Train Model A → Train Model B